# AGCN Presenter v0.1 — Treino QLoRA no Google Colab

Este notebook treina **somente o adapter LoRA** do AGCN Presenter sobre `Qwen/Qwen3-4B`.

- dataset: 93 treino + 7 validação;
- 25 situações congeladas ficam fora do treino;
- thinking desligado;
- saída salva no Google Drive;
- nada é enviado para a `main` automaticamente.

**Antes de rodar:** no Colab escolha GPU em `Ambiente de execução > Alterar tipo de ambiente de execução`.

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NENHUMA')
if not torch.cuda.is_available():
    raise RuntimeError('Ative uma GPU no Colab antes de continuar.')


## 1. Abrir a branch de treinamento
O repositório público não requer senha nem token; a cópia do baseline é reutilizada.


In [ ]:
import os, subprocess, pathlib
repo_dir = pathlib.Path('/content/AGCN-Live-Voice')
branch = 'research/agcn-presenter-training-v1'
if not repo_dir.exists():
    subprocess.run(['git','clone','--depth','1','--branch',branch,'https://github.com/adrse/AGCN-Live-Voice.git',str(repo_dir)], check=True)
os.chdir(repo_dir)
assert subprocess.check_output(['git','branch','--show-current'], text=True).strip() == branch
baseline = pathlib.Path('training/baselines/qwen3_4b_current_v0.1.jsonl')
assert baseline.exists(), 'Conclua o baseline oficial antes do treino.'
import json
baseline_rows = [json.loads(x) for x in baseline.read_text().splitlines() if x.strip()]
assert len(baseline_rows) == 25
print('Branch:', branch, '| baseline:', len(baseline_rows))


## 2. Instalar dependências do treino

In [ ]:
!python -m pip install -q -r training/requirements-lora.txt


## 3. Montar o corpus SFT
O conjunto congelado não entra aqui.

In [ ]:
!python training/scripts/build_sft_corpus.py training/outputs
!python training/scripts/lint_oral_style.py training/datasets/real_live_gold_v0.2.jsonl training/datasets/gold_expansion_v0.3.jsonl
!python training/scripts/dataset_stats.py training/datasets/real_live_gold_v0.2.jsonl training/datasets/gold_expansion_v0.3.jsonl


## 4. Montar o Google Drive
O adapter final vai ser salvo no Drive, não no GitHub.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUTPUT='/content/drive/MyDrive/AGCN/Presenter/agcn-presenter-v0.1-lora'
print(OUTPUT)


## 5. Treinar o primeiro LoRA
Configuração inicial: 4 épocas, LoRA r=16, learning rate 2e-4. Depois vamos comparar com o baseline antes de decidir qualquer integração.

In [ ]:
!python training/scripts/train_lora_qwen3.py \
  --train training/outputs/agcn_presenter_v0.1_train.jsonl \
  --validation training/outputs/agcn_presenter_v0.1_validation.jsonl \
  --output "$OUTPUT" \
  --epochs 4 \
  --learning-rate 0.0002 \
  --lora-r 16 \
  --lora-alpha 32


## 6. Conferir o que foi salvo
O resultado ainda **não entra no programa**. Primeiro ele passa pelas 25 situações congeladas.

In [ ]:
import os, json
print('\n'.join(sorted(os.listdir(OUTPUT))))
summary=os.path.join(OUTPUT,'agcn_training_summary.json')
if os.path.exists(summary):
    print(json.dumps(json.load(open(summary)), indent=2, ensure_ascii=False))
